In [ ]:
from pathlib import Path
from PIL import Image
import subprocess
import random
import shutil

FPS = 12
OUT_DIR = Path("frames")
OUT_DIR.mkdir(exist_ok=True)

assets = {
    "both_quiet": Image.open("both_silent.png").convert("RGB"),
    "left_open": Image.open("left_mouth_open.png").convert("RGB"),
    "right_open": Image.open("right_mouth_open.png").convert("RGB"),
    "left_blink": Image.open("left_blink.png").convert("RGB"),
    "right_blink": Image.open("right_blink.png").convert("RGB"),
}

# Example timeline. Replace this with your generated one.
timeline = (
    ["both_quiet"] * 12 +
    ["left_talking"] * 36 +
    ["both_quiet"] * 10 +
    ["right_talking"] * 36 +
    ["both_quiet"] * 12
)

# Clean old frames
for f in OUT_DIR.glob("frame_*.png"):
    f.unlink()

# Preselect blink frames
blink_frames = set()
for i in range(10, len(timeline), random.randint(30, 55)):
    blink_frames.add(i)

for i, state in enumerate(timeline):
    # Blink override: both mouths closed
    if i in blink_frames:
        img = assets["left_blink"] if random.random() < 0.5 else assets["right_blink"]

    elif state == "left_talking":
        # mouth flap: open every other frame
        img = assets["left_open"] if i % 2 == 0 else assets["both_quiet"]

    elif state == "right_talking":
        img = assets["right_open"] if i % 2 == 0 else assets["both_quiet"]

    else:
        img = assets["both_quiet"]

    img.save(OUT_DIR / f"frame_{i:05d}.png")

# Encode video + audio
cmd = [
    "ffmpeg",
    "-y",
    "-framerate", str(FPS),
    "-i", str(OUT_DIR / "frame_%05d.png"),
    "-i", "anim_audio.mp3",
    "-c:v", "libx264",
    "-pix_fmt", "yuv420p",
    "-c:a", "aac",
    "-shortest",
    "dialogue_animation.mp4",
]

subprocess.run(cmd, check=True)

print("Done: dialogue_animation.mp4")
